In [2]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

2025-09-23 10:54:59 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Vinculaciones adquirencia

In [7]:
# Ingestión información de vinculación
# Se debe esperar hasta el siguiente mes para obtener la vinculación del mes actual
sql = """
SELECT ingestion_year, ingestion_month, ingestion_day, year, month, day, periodo, count(*)
FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
WHERE YEAR BETWEEN 2025 and 2025
  AND MONTH  BETWEEN 1 and 9
  AND DAY BETWEEN 1 and 31
GROUP BY 1,2,3,4,5,6,7
ORDER BY periodo desc, year DESC, month desc, day desc, ingestion_year desc, ingestion_month desc, ingestion_day desc;
"""
helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 2/2 DATAFRAME         ejecutando   12:00:06 PM             

2025-09-23 12:00:19 - [INFO] - 10 filas, 8 columnas, 00:13.1 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 2/2 DATAFRAME         finalizado   12:00:06 PM     00:13.3 
------------------------------------------------------------


,ingestion_year,ingestion_month,ingestion_day,year,month,day,periodo,expr_1
0,2025,9,5,2025,9,5,NaN,20
1,2025,9,5,2025,9,5,202508.0,4198
2,2025,8,8,2025,8,8,202507.0,3817
3,2025,7,8,2025,7,8,202506.0,3092
4,2025,6,18,2025,6,18,202505.0,3198
5,2025,5,6,2025,5,6,202504.0,3244
6,2025,4,4,2025,4,4,202503.0,2827
7,2025,3,11,2025,3,11,202502.0,2735
8,2025,2,7,2025,2,7,202501.0,2361
9,2025,1,11,2025,1,11,202412.0,3325


In [5]:
dict_ult_ing_vinculacion = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_vinculacion

2025-09-23 11:52:58 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
/Users/santlond/anaconda3/envs/env_odbc_py39/lib/python3.9/site-packages/helper/helper.py:421: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-09-23 11:52:59 - [INFO] - Finalizo la busqueda, duracion: 00:00.3, resultado: {'year': 2025, 'month': 9, 'day': 5}


{'year': 2025, 'month': 9, 'day': 5}

In [8]:
# Número de Vinculaciones
sql = """
WITH vinc AS
  (SELECT periodo,
          count(*) AS num_vinc,
          cast(left(cast(periodo AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(periodo AS STRING), 2) AS int) AS mes
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR BETWEEN 2020 AND 2025
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1
   ORDER BY periodo DESC)
SELECT periodo,
       YEAR,
       mes,
       num_vinc,
       sum(num_vinc) over (partition by year order by year, mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as num_vinc_cumsum_year_mes
FROM vinc
order by periodo;
"""
helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 3/3 DATAFRAME          falló n.1   03:38:49 PM             
 3/3 DATAFRAME         finalizado   03:38:49 PM     00:03.1 
------------------------------------------------------------


2025-09-25 15:38:52 - [INFO] - 68 filas, 5 columnas, 00:02.9 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


,periodo,year,mes,num_vinc,num_vinc_cumsum_year_mes
0,202001.0,2020,1,2034,2034
1,202002.0,2020,2,2436,4470
2,202003.0,2020,3,3026,7496
3,202004.0,2020,4,1059,8555
4,202005.0,2020,5,1967,10522
...,...,...,...,...,...
63,202504.0,2025,4,3244,11167
64,202505.0,2025,5,3198,14365
65,202506.0,2025,6,3092,17457
66,202507.0,2025,7,3817,21274


# Uso adquirencia

Se calcula por año, un comercio usa adquirencia [activo] cuando al menos hace 1 trx en lo que va corrido de un año.

Por ejemplo, un comercio que se vinculo en febrero de 2025 estará activo si realiza al menos 1 trx en el año 2025.

Se elegirá la estrategia de seleccionar la primera trx en el año.

## Análisis ingestión compras tabla transaccional adquirencia

In [14]:
# Se deben esperar dos días para que se ingeste la información completa de las transacciones de un día en particular
# Para obtener la información de las compras del día jueves y viernes se debe espear hasta la sgte semana
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          left(cast(f_trx as string), 10) as f_trx,
          count(*) AS num_compras
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
   WHERE YEAR IN (2025)
     AND MONTH BETWEEN 8 AND 9
     AND DAY BETWEEN 1 AND 31
     and tipo_trx = "Purchase"
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
helper.obtener_dataframe(sql)[177:220]

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 8/8 DATAFRAME         ejecutando   04:00:53 PM             

2025-09-25 16:01:15 - [INFO] - 1,413 filas, 10 columnas, 00:21.9 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 8/8 DATAFRAME         finalizado   04:00:53 PM     00:22.1 
------------------------------------------------------------


,year,month,day,f_trx,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
177,2025,9,2,2025-09-01,1720533,1761572,1720576,2,0.9767,0.9767
178,2025,9,1,2025-09-01,43,1761572,43,1,0.0000,0.0000
179,2025,9,25,2025-08-31,3,2103965,2103965,17,0.0000,1.0000
180,2025,9,23,2025-08-31,2,2103965,2103962,16,0.0000,1.0000
181,2025,9,19,2025-08-31,2,2103965,2103960,15,0.0000,1.0000
182,2025,9,18,2025-08-31,1,2103965,2103958,14,0.0000,1.0000
183,2025,9,17,2025-08-31,1,2103965,2103957,13,0.0000,1.0000
184,2025,9,16,2025-08-31,1,2103965,2103956,12,0.0000,1.0000
185,2025,9,15,2025-08-31,3,2103965,2103955,11,0.0000,1.0000
186,2025,9,12,2025-08-31,4,2103965,2103952,10,0.0000,1.0000


In [36]:
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2022-09-01' # MODIFICAR.
fecha_final = '2022-12-21' # Modificar.
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='D')

# Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
pri_dia_part = fechas[-1] + relativedelta(days=1)
pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
ult_dia_part = fechas[-1] + relativedelta(days=10)
ult_dia_part = ult_dia_part.date().isoformat()
fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config

,fechas,year,month,day
0,2022-09-01,2022,9,1
1,2022-09-02,2022,9,2
2,2022-09-03,2022,9,3
3,2022-09-04,2022,9,4
4,2022-09-05,2022,9,5
...,...,...,...,...
117,2022-12-27,2022,12,27
118,2022-12-28,2022,12,28
119,2022-12-29,2022,12,29
120,2022-12-30,2022,12,30


In [ ]:
# # Crear tabla que almacenará la información
# sql_drop = """DROP TABLE IF EXISTS proceso_bluekai.mdo_adquirencia_trxs PURGE;"""
# helper.ejecutar_consulta(sql_drop)

# sql = """
# CREATE TABLE proceso_bluekai.mdo_adquirencia_trxs  (
#                 cod_unico VARCHAR,
#                 f_trx TIMESTAMP,
#                 num_trxs BIGINT,
#                 mnt_total_trxs DECIMAL(38,2)
#                 )
#             PARTITIONED BY 
#             (
#             YEAR INT,
#             MES INT,
#             DIA INT
#             )
# STORED AS PARQUET
# TBLPROPERTIES ('transactional' = 'false');
# """
# helper.ejecutar_consulta(sql)

# sql_compute = """COMPUTE STATS proceso_bluekai.mdo_adquirencia_trxs;"""

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 53/53      DROP proceso_bluekai.mdo_adquirencia_trxs   finalizado   07:47:50 AM     00:01.2 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 54/54    CREATE proceso_bluekai.mdo_adquirencia_trxs   finalizado   07:47:51 AM     00:00.1 
---------------------------------------------------------------------------------------------


In [37]:
# Iterar para obtener las trxs por cliente
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month), '-', str(row.day))
    print('')
    print('Obteniendo transacciones adquirencia de los comercios')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    SELECT cod_unico,
       f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs,
        """ + str(row.year) + """ AS YEAR,
        """ + str(row.month) + """ AS MES,
        """ + str(row.day) + """ AS DIA
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = """ + str(row.year) + """
     AND MONTH = """ + str(row.month) + """
     AND DAY = """ + str(row.day) + """
     AND tipo_trx = "Purchase"
    GROUP BY 1,
            2;"""
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando transacciones adquirencia de los comercios')
    print('')

    sql = """
    INSERT INTO proceso_bluekai.mdo_adquirencia_trxs PARTITION (YEAR, MES, DIA)
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           YEAR,
           MES,
           DIA
    FROM proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql)
    print('')

##################################################

Exrayendo datos de las particiones:  2022 - 9 - 1

Obteniendo transacciones adquirencia de los comercios

-------------------------------------------------------------------------------------------------
     i       tipo                   nombre                   estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 3981/3981      DROP    proceso.mdo_adquirencia_trxs_temp    falló n.1   11:52:35 AM             
 3981/3981      DROP    proceso.mdo_adquirencia_trxs_temp   finalizado   11:52:35 AM     00:01.7 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
     i       tipo                   nombre                   estado     hora_inicio   duracion   
----------------------------------------------------------

In [ ]:
# Lógica para construir el uso

# Ejemplo manual para el periodo 202508

# Previamente crear tabla para almacenar los comercios con trxs por periodo

# Obtener las vinculaciones del mes correspondiente
sql = """
CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
select codigo_unico, 202508 AS periodo
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR BETWEEN 2020 AND 2025
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo in (202501, 202502, 202503, 202504, 202505, 202506, 202507, 202508);
"""

# Obtener las transacciones seis meses hacia atrás
sql = """
CREATE TABLE prceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
select distinct cod_unico
FROM proceso_bluekai.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2025 and 2025
and mes BETWEEN 3 and 9
and dia BETWEEN 1 AND 31
and f_trx >= '2025-03-01';
"""
# Obtener vinculaciones que hayan hecho transacciones
"""
CREATE TABLE proceso.mdo_adquirencia_vinculaciones_con_trxs_temp STORED AS PARQUET AS
SELECT a.codigo_unico, a.periodo
FROM proceso.mdo_adquirencia_vinculaciones_temp as a
INNER JOIN proceso.mdo_adquirencia_trxs_temp as b on a.codigo_unico = b.cod_unico;
"""

# Insertar información en tabla


In [ ]:
# Crear tabla que almacenará para cada año la primera fecha de trx de un comercio
# Se debe hacer mes por mes
sql = """
SELECT cod_unico, f_trx
FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
WHERE YEAR BETWEEN 2025 and 2025
  AND MONTH BETWEEN 8 and 8
  AND DAY BETWEEN 1 AND 31
  AND tipo_trx = "Purchase"
  AND estado_trx = 'Cleared';
"""